# SQL Operators Assignment — Employee Dataset
**Dataset:** 150 employee records, 29 columns
**Goal:** Load the dataset into a PostgreSQL table named `employees`, then answer all 50
assignment questions using SQL — executed from Python (psycopg2 + SQLAlchemy) inside this
Jupyter notebook.

**Topics covered:** Arithmetic, Comparison, Logical, `LIKE`, `IN`, `BETWEEN`, `IS NULL`, `IS NOT NULL`.

> Run this notebook top to bottom in VS Code (Jupyter extension). Database credentials are
> loaded securely from a `.env` file (not hardcoded, not committed to git) — see
> `.env.example` and `.gitignore` alongside this notebook.
>
> Install dependencies first: `pip install psycopg2-binary sqlalchemy pandas openpyxl python-dotenv`


## 1. Imports

In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
from dotenv import load_dotenv



## 2. Database connection (credentials from `.env`)




In [2]:
load_dotenv()  # reads variables from a .env file in the same folder as this notebook

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")


CONN_STRING = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(CONN_STRING)

# Raw psycopg2 connection (used for DDL statements)
conn = psycopg2.connect(
    host=DB_HOST,
    port=DB_PORT,
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
)
conn.autocommit = True
cursor = conn.cursor()

print("Connected to PostgreSQL:", DB_NAME, "as", DB_USER)


Connected to PostgreSQL: employee_db as postgres


## 3. Load the dataset from Excel



In [3]:
CSV_PATH = "Employees.csv"   # path to the assignment csv

df = pd.read_csv(CSV_PATH)

# phone / emergency_contact come in as floats (e.g. 9869202768.0) because of blank cells;
# convert to clean nullable strings without altering the real digits.
for col in ["phone", "emergency_contact"]:
    df[col] = df[col].apply(lambda x: str(int(x)) if pd.notna(x) else None)

df["join_date"] = pd.to_datetime(df["join_date"]).dt.date

print(df.shape)
df.head()


(150, 29)


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
3,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
4,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No


## 4. Create the `employees` table

Explicit `CREATE TABLE` with types matching each of the 29 columns, wrapped in Python and
executed through `psycopg2`.


In [4]:
CREATE_TABLE_SQL = """
DROP TABLE IF EXISTS employees;

CREATE TABLE employees (
    employee_id            INTEGER PRIMARY KEY,
    employee_code          VARCHAR(20),
    first_name              VARCHAR(50),
    last_name               VARCHAR(50),
    gender                   VARCHAR(20),
    age                      INTEGER,
    city                     VARCHAR(50),
    department               VARCHAR(50),
    job_title                VARCHAR(50),
    employment_type          VARCHAR(20),
    join_date                DATE,
    years_experience         INTEGER,
    monthly_salary           NUMERIC(12,2),
    annual_bonus             NUMERIC(12,2),
    performance_rating       NUMERIC(3,1),
    projects_completed       INTEGER,
    leave_days_taken         INTEGER,
    overtime_hours           INTEGER,
    remote_worker             VARCHAR(5),
    employment_status         VARCHAR(20),
    education_level           VARCHAR(20),
    email                     VARCHAR(100),
    phone                     VARCHAR(20),
    emergency_contact         VARCHAR(20),
    manager_name              VARCHAR(50),
    certification              VARCHAR(50),
    work_shift                 VARCHAR(20),
    performance_category       VARCHAR(20),
    promotion_eligible          VARCHAR(5)
);
"""

cursor.execute(CREATE_TABLE_SQL)
print("Table 'employees' created.")


Table 'employees' created.


## 5. Load the DataFrame into PostgreSQL

In [5]:
df.to_sql("employees", engine, if_exists="append", index=False)
print("Inserted", len(df), "rows into employees.")


Inserted 150 rows into employees.


### Sanity check

In [6]:
check = pd.read_sql_query("SELECT COUNT(*) AS row_count FROM employees;", engine)
check


,row_count
0,150


## 6. Assignment Questions (1–50)




In [7]:
def run_query(sql: str) -> pd.DataFrame:
    """Run a SQL string against the employees table and return the result as a DataFrame."""
    return pd.read_sql_query(sql, engine)


### Arithmetic Operators

**Q1. Display employee_id, monthly_salary, and monthly_salary + 10000 as increased_salary.**

In [8]:
query_1 = """
SELECT employee_id, monthly_salary, monthly_salary + 10000 AS increased_salary
FROM employees;
"""
result_1 = run_query(query_1)
print(f"Rows returned: {len(result_1)}")
result_1.head(20)


Rows returned: 150


,employee_id,monthly_salary,increased_salary
0,1,105000.0,115000.0
1,2,145000.0,155000.0
2,3,70000.0,80000.0
3,4,187500.0,197500.0
4,5,200000.0,210000.0
5,6,55000.0,65000.0
6,7,35000.0,45000.0
7,8,195000.0,205000.0
8,9,192500.0,202500.0
9,10,240000.0,250000.0


**Q2. Calculate reduced_salary by subtracting 5000 from monthly_salary.**

In [9]:
query_2 = """
SELECT employee_id, monthly_salary, monthly_salary - 5000 AS reduced_salary
FROM employees;
"""
result_2 = run_query(query_2)
print(f"Rows returned: {len(result_2)}")
result_2.head(20)


Rows returned: 150


,employee_id,monthly_salary,reduced_salary
0,1,105000.0,100000.0
1,2,145000.0,140000.0
2,3,70000.0,65000.0
3,4,187500.0,182500.0
4,5,200000.0,195000.0
5,6,55000.0,50000.0
6,7,35000.0,30000.0
7,8,195000.0,190000.0
8,9,192500.0,187500.0
9,10,240000.0,235000.0


**Q3. Calculate annual_salary by multiplying monthly_salary by 12.**

In [10]:
query_3 = """
SELECT employee_id, monthly_salary, monthly_salary * 12 AS annual_salary
FROM employees;
"""
result_3 = run_query(query_3)
print(f"Rows returned: {len(result_3)}")
result_3.head(20)


Rows returned: 150


,employee_id,monthly_salary,annual_salary
0,1,105000.0,1260000.0
1,2,145000.0,1740000.0
2,3,70000.0,840000.0
3,4,187500.0,2250000.0
4,5,200000.0,2400000.0
5,6,55000.0,660000.0
6,7,35000.0,420000.0
7,8,195000.0,2340000.0
8,9,192500.0,2310000.0
9,10,240000.0,2880000.0


**Q4. Calculate average_monthly_bonus by dividing annual_bonus by 12.**

In [11]:
query_4 = """
SELECT employee_id, annual_bonus, annual_bonus / 12.0 AS average_monthly_bonus
FROM employees;
"""
result_4 = run_query(query_4)
print(f"Rows returned: {len(result_4)}")
result_4.head(20)


Rows returned: 150


,employee_id,annual_bonus,average_monthly_bonus
0,1,15750.0,1312.500000
1,2,14500.0,1208.333333
2,3,8400.0,700.000000
3,4,15000.0,1250.000000
4,5,24000.0,2000.000000
5,6,5500.0,458.333333
6,7,3500.0,291.666667
7,8,23400.0,1950.000000
8,9,23100.0,1925.000000
9,10,19200.0,1600.000000


**Q5. Display age and age % 2 as remainder.**

In [12]:
query_5 = """
SELECT employee_id, age, age %% 2 AS remainder
FROM employees;
"""
result_5 = run_query(query_5)
print(f"Rows returned: {len(result_5)}")
result_5.head(20)


Rows returned: 150


,employee_id,age,remainder
0,1,53,1
1,2,46,0
2,3,58,0
3,4,51,1
4,5,59,1
5,6,48,0
6,7,43,1
7,8,58,0
8,9,41,1
9,10,24,0


**Q6. Calculate total_compensation as (monthly_salary * 12) + annual_bonus.**

In [13]:
query_6 = """
SELECT employee_id, monthly_salary,annual_bonus,(monthly_salary * 12) + annual_bonus AS total_compensation
FROM employees;
"""
result_6 = run_query(query_6)
print(f"Rows returned: {len(result_6)}")
result_6.head(20)


Rows returned: 150


,employee_id,monthly_salary,annual_bonus,total_compensation
0,1,105000.0,15750.0,1275750.0
1,2,145000.0,14500.0,1754500.0
2,3,70000.0,8400.0,848400.0
3,4,187500.0,15000.0,2265000.0
4,5,200000.0,24000.0,2424000.0
5,6,55000.0,5500.0,665500.0
6,7,35000.0,3500.0,423500.0
7,8,195000.0,23400.0,2363400.0
8,9,192500.0,23100.0,2333100.0
9,10,240000.0,19200.0,2899200.0


### Comparison Operators

**Q7. Find employees whose monthly_salary is greater than 100000.**

In [14]:
query_7 = """
SELECT *
FROM employees
WHERE monthly_salary > 100000;
"""
result_7 = run_query(query_7)
print(f"Rows returned: {len(result_7)}")
result_7.head(20)


Rows returned: 97


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
3,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
4,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
5,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
6,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
7,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
8,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
9,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes


**Q8. Find employees whose age is less than 30.**

In [15]:
query_8 = """
SELECT *
FROM employees
WHERE age < 30;
"""
result_8 = run_query(query_8)
print(f"Rows returned: {len(result_8)}")
result_8.head(20)


Rows returned: 27


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
1,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
2,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
3,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes
4,22,EMP0022,Aarav,Maharjan,Female,28,Butwal,HR,HR Manager,Full-Time,...,On Leave,PhD,aarav.maharjan22@company.com,9824263193,9853794068,Rita Thapa,Power BI,Flexible,Good,Yes
5,34,EMP0034,Binita,Poudel,Female,26,Chitwan,Sales,Sales Manager,Intern,...,On Leave,PhD,binita.poudel34@company.com,9838549945,NaN,Mina Rai,AWS,Flexible,Excellent,No
6,37,EMP0037,Kusum,Sharma,Other,26,Kathmandu,Analytics,Data Analyst,Intern,...,Inactive,Bachelor,kusum.sharma37@company.com,9849830184,9814996960,Prakash Karki,SQL,Evening,Average,No
7,38,EMP0038,Sanjay,Adhikari,Other,22,Kathmandu,Sales,Business Development Officer,Contract,...,Inactive,Master,NaN,9816199453,9819546277,Sanjay Sharma,PMP,Evening,Good,Yes
8,43,EMP0043,Amit,Sharma,Female,23,Chitwan,Finance,Accountant,Intern,...,Active,Master,amit.sharma43@company.com,9817143479,9882701840,Mina Rai,AWS,Day,Excellent,No
9,47,EMP0047,Anish,Thapa,Female,21,Bhaktapur,IT,QA Engineer,Full-Time,...,Active,Master,anish.thapa47@company.com,9861155566,9895030117,Rita Thapa,NaN,Evening,Good,Yes


**Q9. Find employees with performance_rating greater than or equal to 4.5.**

In [16]:
query_9 = """
SELECT *
FROM employees
WHERE performance_rating >= 4.5;
"""
result_9 = run_query(query_9)
print(f"Rows returned: {len(result_9)}")
result_9.head(20)


Rows returned: 32


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
1,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
2,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
3,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes
4,20,EMP0020,Suman,Gurung,Other,58,Bhaktapur,IT,Data Engineer,Contract,...,On Leave,Bachelor,suman.gurung20@company.com,9824082320,9821480544,Amit Pandey,Python,Flexible,Average,Yes
5,21,EMP0021,Sneha,Rai,Male,39,Butwal,HR,HR Manager,Part-Time,...,Inactive,Bachelor,sneha.rai21@company.com,9896943854,9874426133,Amit Pandey,Power BI,Day,Good,Yes
6,41,EMP0041,Anish,Adhikari,Female,31,Chitwan,Sales,Sales Manager,Contract,...,Active,PhD,anish.adhikari41@company.com,9836393021,9816860253,Rita Thapa,AWS,Day,Excellent,Yes
7,43,EMP0043,Amit,Sharma,Female,23,Chitwan,Finance,Accountant,Intern,...,Active,Master,amit.sharma43@company.com,9817143479,9882701840,Mina Rai,AWS,Day,Excellent,No
8,47,EMP0047,Anish,Thapa,Female,21,Bhaktapur,IT,QA Engineer,Full-Time,...,Active,Master,anish.thapa47@company.com,9861155566,9895030117,Rita Thapa,NaN,Evening,Good,Yes
9,65,EMP0065,Roshani,Shrestha,Female,34,Biratnagar,Analytics,Data Analyst,Full-Time,...,On Leave,Diploma,roshani.shrestha65@company.com,9848110898,9850083036,Prakash Karki,NaN,Day,Excellent,Yes


**Q10. Find employees whose years_experience is less than or equal to 5.**

In [17]:
query_10 = """
SELECT *
FROM employees
WHERE years_experience <= 5;
"""
result_10 = run_query(query_10)
print(f"Rows returned: {len(result_10)}")
result_10.head(20)


Rows returned: 80


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
1,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
2,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
3,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
4,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
5,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
6,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
7,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
8,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
9,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,NaN,Mina Rai,AWS,Flexible,Average,No


**Q11. Find employees whose department is equal to IT.**

In [18]:
query_11 = """
SELECT *
FROM employees
WHERE department = 'IT';
"""
result_11 = run_query(query_11)
print(f"Rows returned: {len(result_11)}")
result_11.head(20)


Rows returned: 22


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
2,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
3,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
4,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
5,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
6,20,EMP0020,Suman,Gurung,Other,58,Bhaktapur,IT,Data Engineer,Contract,...,On Leave,Bachelor,suman.gurung20@company.com,9824082320,9821480544,Amit Pandey,Python,Flexible,Average,Yes
7,26,EMP0026,Nabin,Karki,Other,52,Lalitpur,IT,Software Engineer,Contract,...,Inactive,Diploma,nabin.karki26@company.com,9845843496,9885017167,Sanjay Sharma,NaN,Evening,Good,No
8,27,EMP0027,Aarya,Karki,Other,35,Biratnagar,IT,Software Engineer,Full-Time,...,Active,Master,aarya.karki27@company.com,9894579966,9859584636,Amit Pandey,SQL,Evening,Excellent,No
9,31,EMP0031,Sabina,Gurung,Other,34,Bhaktapur,IT,QA Engineer,Part-Time,...,Active,Master,sabina.gurung31@company.com,9885012599,9853722847,Rita Thapa,AWS,Flexible,Excellent,Yes


**Q12. Find employees whose employment_status is not equal to Active.**

In [19]:
query_12 = """
SELECT *
FROM employees
WHERE employment_status <> 'Active';
"""
result_12 = run_query(query_12)
print(f"Rows returned: {len(result_12)}")
result_12.head(20)


Rows returned: 97


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
1,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
2,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
3,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
4,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
5,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
6,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes
7,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
8,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
9,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No


### Logical Operators

**Q13. Find employees from Kathmandu AND monthly_salary greater than 100000.**

In [20]:
query_13 = """
SELECT *
FROM employees
WHERE city = 'Kathmandu' AND monthly_salary > 100000;
"""
result_13 = run_query(query_13)
print(f"Rows returned: {len(result_13)}")
result_13.head(20)


Rows returned: 18


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
2,33,EMP0033,Aarav,Rai,Female,35,Kathmandu,Sales,Sales Executive,Contract,...,Inactive,PhD,aarav.rai33@company.com,9878214548,9815405350,Rita Thapa,Power BI,Evening,Excellent,Yes
3,37,EMP0037,Kusum,Sharma,Other,26,Kathmandu,Analytics,Data Analyst,Intern,...,Inactive,Bachelor,kusum.sharma37@company.com,9849830184,9814996960,Prakash Karki,SQL,Evening,Average,No
4,38,EMP0038,Sanjay,Adhikari,Other,22,Kathmandu,Sales,Business Development Officer,Contract,...,Inactive,Master,NaN,9816199453,9819546277,Sanjay Sharma,PMP,Evening,Good,Yes
5,48,EMP0048,Ramesh,Rai,Other,49,Kathmandu,Analytics,Data Analyst,Part-Time,...,Inactive,PhD,ramesh.rai48@company.com,9832568994,9823644673,Prakash Karki,PMP,Flexible,Good,Yes
6,53,EMP0053,Sneha,Khatri,Female,27,Kathmandu,Finance,Financial Analyst,Contract,...,Active,Master,sneha.khatri53@company.com,9875191287,9855752567,Amit Pandey,PMP,Evening,Good,Yes
7,63,EMP0063,Suman,Shrestha,Other,55,Kathmandu,Sales,Sales Manager,Contract,...,Inactive,Diploma,suman.shrestha63@company.com,9836063000,9884127728,Rita Thapa,AWS,Evening,Excellent,Yes
8,68,EMP0068,Roshani,Maharjan,Female,43,Kathmandu,IT,Software Engineer,Intern,...,On Leave,Master,roshani.maharjan68@company.com,9871761686,NaN,Rita Thapa,NaN,Evening,Average,Yes
9,72,EMP0072,Amit,Sharma,Other,38,Kathmandu,Finance,Finance Officer,Part-Time,...,On Leave,PhD,amit.sharma72@company.com,9871713926,9879496093,Prakash Karki,NaN,Evening,Excellent,Yes


**Q14. Find employees from Kathmandu OR Pokhara.**

In [21]:
query_14 = """
SELECT *
FROM employees
WHERE city = 'Kathmandu' OR city = 'Pokhara';
"""
result_14 = run_query(query_14)
print(f"Rows returned: {len(result_14)}")
result_14.head(20)


Rows returned: 42


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
2,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
3,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes
4,28,EMP0028,Amit,Sharma,Female,31,Pokhara,HR,HR Manager,Part-Time,...,On Leave,PhD,amit.sharma28@company.com,9813798947,9846356183,Sanjay Sharma,Power BI,Evening,Good,No
5,30,EMP0030,Anish,Basnet,Male,53,Pokhara,Analytics,Data Analyst,Part-Time,...,On Leave,Diploma,anish.basnet30@company.com,9852672583,9898658609,Mina Rai,AWS,Flexible,Excellent,Yes
6,32,EMP0032,Pratigya,Karki,Male,57,Pokhara,Analytics,Data Analyst,Full-Time,...,Active,Master,pratigya.karki32@company.com,9882520415,9857868867,Prakash Karki,Python,Evening,Average,No
7,33,EMP0033,Aarav,Rai,Female,35,Kathmandu,Sales,Sales Executive,Contract,...,Inactive,PhD,aarav.rai33@company.com,9878214548,9815405350,Rita Thapa,Power BI,Evening,Excellent,Yes
8,37,EMP0037,Kusum,Sharma,Other,26,Kathmandu,Analytics,Data Analyst,Intern,...,Inactive,Bachelor,kusum.sharma37@company.com,9849830184,9814996960,Prakash Karki,SQL,Evening,Average,No
9,38,EMP0038,Sanjay,Adhikari,Other,22,Kathmandu,Sales,Business Development Officer,Contract,...,Inactive,Master,NaN,9816199453,9819546277,Sanjay Sharma,PMP,Evening,Good,Yes


**Q15. Find employees from IT OR Analytics with performance_rating greater than 4.**

In [22]:
query_15 = """
SELECT *
FROM employees
WHERE (department = 'IT' OR department = 'Analytics')
  AND performance_rating > 4;
"""
result_15 = run_query(query_15)
print(f"Rows returned: {len(result_15)}")
result_15.head(20)


Rows returned: 17


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
2,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
3,20,EMP0020,Suman,Gurung,Other,58,Bhaktapur,IT,Data Engineer,Contract,...,On Leave,Bachelor,suman.gurung20@company.com,9824082320,9821480544,Amit Pandey,Python,Flexible,Average,Yes
4,26,EMP0026,Nabin,Karki,Other,52,Lalitpur,IT,Software Engineer,Contract,...,Inactive,Diploma,nabin.karki26@company.com,9845843496,9885017167,Sanjay Sharma,NaN,Evening,Good,No
5,46,EMP0046,Ramesh,Maharjan,Male,36,Chitwan,IT,Software Engineer,Part-Time,...,Inactive,Diploma,ramesh.maharjan46@company.com,NaN,9833557996,Sanjay Sharma,NaN,Day,Average,No
6,47,EMP0047,Anish,Thapa,Female,21,Bhaktapur,IT,QA Engineer,Full-Time,...,Active,Master,anish.thapa47@company.com,9861155566,9895030117,Rita Thapa,NaN,Evening,Good,Yes
7,50,EMP0050,Sujan,Lama,Female,33,Butwal,IT,QA Engineer,Contract,...,Inactive,PhD,sujan.lama50@company.com,9892119212,9851221953,Sanjay Sharma,NaN,Evening,Excellent,No
8,51,EMP0051,Nabin,Poudel,Other,60,Lalitpur,Analytics,Data Analyst,Intern,...,On Leave,Bachelor,nabin.poudel51@company.com,9867614636,NaN,Amit Pandey,Power BI,Day,Excellent,Yes
9,65,EMP0065,Roshani,Shrestha,Female,34,Biratnagar,Analytics,Data Analyst,Full-Time,...,On Leave,Diploma,roshani.shrestha65@company.com,9848110898,9850083036,Prakash Karki,NaN,Day,Excellent,Yes


**Q16. Find employees who are NOT remote workers.**

In [23]:
query_16 = """
SELECT *
FROM employees
WHERE remote_worker = 'No';
"""
result_16 = run_query(query_16)
print(f"Rows returned: {len(result_16)}")
result_16.head(20)


Rows returned: 75


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
3,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
4,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
5,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
6,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
7,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
8,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
9,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes


**Q17. Find employees with age below 40 AND years_experience above 5 AND employment_status = Active.**

In [24]:
query_17 = """
SELECT *
FROM employees
WHERE age < 40
  AND years_experience > 5
  AND employment_status = 'Active';
"""
result_17 = run_query(query_17)
print(f"Rows returned: {len(result_17)}")
result_17.head(20)


Rows returned: 14


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes
1,27,EMP0027,Aarya,Karki,Other,35,Biratnagar,IT,Software Engineer,Full-Time,...,Active,Master,aarya.karki27@company.com,9894579966,9859584636,Amit Pandey,SQL,Evening,Excellent,No
2,53,EMP0053,Sneha,Khatri,Female,27,Kathmandu,Finance,Financial Analyst,Contract,...,Active,Master,sneha.khatri53@company.com,9875191287,9855752567,Amit Pandey,PMP,Evening,Good,Yes
3,60,EMP0060,Roshani,Shrestha,Other,27,Butwal,IT,Data Engineer,Part-Time,...,Active,Diploma,roshani.shrestha60@company.com,9864888845,9859795676,Prakash Karki,PMP,Evening,Excellent,No
4,62,EMP0062,Prakash,Thapa,Female,39,Lalitpur,HR,HR Manager,Part-Time,...,Active,PhD,prakash.thapa62@company.com,9859264105,9887444334,Sanjay Sharma,NaN,Evening,Excellent,No
5,77,EMP0077,Roshani,Lama,Other,31,Butwal,Sales,Sales Executive,Intern,...,Active,PhD,roshani.lama77@company.com,9818242695,9827301329,Mina Rai,Power BI,Flexible,Excellent,Yes
6,96,EMP0096,Pratigya,Shrestha,Female,35,Biratnagar,Marketing,Brand Manager,Contract,...,Active,Master,pratigya.shrestha96@company.com,9899228205,9813690274,Prakash Karki,NaN,Day,Good,No
7,99,EMP0099,Binita,Karki,Male,36,Butwal,IT,QA Engineer,Intern,...,Active,Bachelor,binita.karki99@company.com,9878656923,9819618499,Mina Rai,PMP,Day,Excellent,Yes
8,101,EMP0101,Mina,Lama,Female,31,Butwal,Analytics,BI Analyst,Part-Time,...,Active,Diploma,mina.lama101@company.com,9891035867,9854449566,Amit Pandey,Python,Flexible,Excellent,Yes
9,140,EMP0140,Rahul,Poudel,Female,36,Kathmandu,Finance,Financial Analyst,Part-Time,...,Active,Master,rahul.poudel140@company.com,9835349138,9869524317,Rita Thapa,Power BI,Flexible,Excellent,No


**Q18. Find Full-Time employees with salary above 80000 OR performance_rating above 4.5.**

In [25]:
query_18 = """
SELECT *
FROM employees
WHERE employment_type = 'Full-Time'
  AND (monthly_salary > 80000 OR performance_rating > 4.5);
"""
result_18 = run_query(query_18)
print(f"Rows returned: {len(result_18)}")
result_18.head(20)


Rows returned: 28


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
1,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
2,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,NaN,Mina Rai,AWS,Flexible,Average,No
3,22,EMP0022,Aarav,Maharjan,Female,28,Butwal,HR,HR Manager,Full-Time,...,On Leave,PhD,aarav.maharjan22@company.com,9824263193,9853794068,Rita Thapa,Power BI,Flexible,Good,Yes
4,27,EMP0027,Aarya,Karki,Other,35,Biratnagar,IT,Software Engineer,Full-Time,...,Active,Master,aarya.karki27@company.com,9894579966,9859584636,Amit Pandey,SQL,Evening,Excellent,No
5,32,EMP0032,Pratigya,Karki,Male,57,Pokhara,Analytics,Data Analyst,Full-Time,...,Active,Master,pratigya.karki32@company.com,9882520415,9857868867,Prakash Karki,Python,Evening,Average,No
6,39,EMP0039,Roshani,Maharjan,Female,50,Bhaktapur,Finance,Financial Analyst,Full-Time,...,Inactive,Diploma,roshani.maharjan39@company.com,9843449924,9889951966,Mina Rai,NaN,Flexible,Good,No
7,47,EMP0047,Anish,Thapa,Female,21,Bhaktapur,IT,QA Engineer,Full-Time,...,Active,Master,anish.thapa47@company.com,9861155566,9895030117,Rita Thapa,NaN,Evening,Good,Yes
8,59,EMP0059,Ramesh,Rai,Other,37,Chitwan,Finance,Finance Officer,Full-Time,...,On Leave,Master,ramesh.rai59@company.com,9882437717,9871221053,Amit Pandey,SQL,Day,Good,Yes
9,64,EMP0064,Bikash,Tamang,Male,51,Bhaktapur,Analytics,Data Scientist,Full-Time,...,On Leave,Master,bikash.tamang64@company.com,9856325053,9840377727,Amit Pandey,NaN,Evening,Good,No


### LIKE Operator

**Q19. Find employees whose first_name starts with A.**

In [27]:
query_19 = """
SELECT *
FROM employees
WHERE first_name LIKE 'A%%';
"""
result_19 = run_query(query_19)
print(f"Rows returned: {len(result_19)}")
result_19.head(20)


Rows returned: 37


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
1,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
2,22,EMP0022,Aarav,Maharjan,Female,28,Butwal,HR,HR Manager,Full-Time,...,On Leave,PhD,aarav.maharjan22@company.com,9824263193,9853794068,Rita Thapa,Power BI,Flexible,Good,Yes
3,27,EMP0027,Aarya,Karki,Other,35,Biratnagar,IT,Software Engineer,Full-Time,...,Active,Master,aarya.karki27@company.com,9894579966,9859584636,Amit Pandey,SQL,Evening,Excellent,No
4,28,EMP0028,Amit,Sharma,Female,31,Pokhara,HR,HR Manager,Part-Time,...,On Leave,PhD,amit.sharma28@company.com,9813798947,9846356183,Sanjay Sharma,Power BI,Evening,Good,No
5,30,EMP0030,Anish,Basnet,Male,53,Pokhara,Analytics,Data Analyst,Part-Time,...,On Leave,Diploma,anish.basnet30@company.com,9852672583,9898658609,Mina Rai,AWS,Flexible,Excellent,Yes
6,33,EMP0033,Aarav,Rai,Female,35,Kathmandu,Sales,Sales Executive,Contract,...,Inactive,PhD,aarav.rai33@company.com,9878214548,9815405350,Rita Thapa,Power BI,Evening,Excellent,Yes
7,40,EMP0040,Aarav,Pandey,Male,34,Chitwan,Sales,Business Development Officer,Part-Time,...,Inactive,Bachelor,aarav.pandey40@company.com,9867943072,9887892505,Prakash Karki,SQL,Flexible,Good,Yes
8,41,EMP0041,Anish,Adhikari,Female,31,Chitwan,Sales,Sales Manager,Contract,...,Active,PhD,anish.adhikari41@company.com,9836393021,9816860253,Rita Thapa,AWS,Day,Excellent,Yes
9,43,EMP0043,Amit,Sharma,Female,23,Chitwan,Finance,Accountant,Intern,...,Active,Master,amit.sharma43@company.com,9817143479,9882701840,Mina Rai,AWS,Day,Excellent,No


**Q20. Find employees whose first_name ends with a.**

In [29]:
query_20 = """
SELECT *
FROM employees
WHERE first_name LIKE '%%a';
"""
result_20 = run_query(query_20)
print(f"Rows returned: {len(result_20)}")
result_20.head(20)


Rows returned: 59


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
1,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
2,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
3,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
4,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
5,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes
6,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
7,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
8,18,EMP0018,Mina,Adhikari,Male,41,Butwal,Finance,Financial Analyst,Contract,...,On Leave,Master,mina.adhikari18@company.com,9830246318,9863562849,Amit Pandey,NaN,Evening,Good,No
9,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes


**Q21. Find employees whose first_name contains the letter i.**

In [31]:
query_21 = """
SELECT *
FROM employees
WHERE first_name LIKE '%%i%%';
"""
result_21 = run_query(query_21)
print(f"Rows returned: {len(result_21)}")
result_21.head(20)


Rows returned: 75


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
1,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
2,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
3,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
4,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes
5,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
6,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
7,18,EMP0018,Mina,Adhikari,Male,41,Butwal,Finance,Financial Analyst,Contract,...,On Leave,Master,mina.adhikari18@company.com,9830246318,9863562849,Amit Pandey,NaN,Evening,Good,No
8,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes
9,25,EMP0025,Pratigya,Poudel,Other,49,Bhaktapur,Finance,Accountant,Full-Time,...,Inactive,PhD,pratigya.poudel25@company.com,9849388796,9887093109,Prakash Karki,Power BI,Flexible,Good,No


**Q22. Find employees whose last_name starts with S.**

In [32]:
query_22 = """
SELECT *
FROM employees
WHERE last_name LIKE 'S%%';
"""
result_22 = run_query(query_22)
print(f"Rows returned: {len(result_22)}")
result_22.head(20)


Rows returned: 14


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,28,EMP0028,Amit,Sharma,Female,31,Pokhara,HR,HR Manager,Part-Time,...,On Leave,PhD,amit.sharma28@company.com,9813798947,9846356183,Sanjay Sharma,Power BI,Evening,Good,No
1,37,EMP0037,Kusum,Sharma,Other,26,Kathmandu,Analytics,Data Analyst,Intern,...,Inactive,Bachelor,kusum.sharma37@company.com,9849830184,9814996960,Prakash Karki,SQL,Evening,Average,No
2,42,EMP0042,Pratigya,Sharma,Female,56,Lalitpur,Operations,Operations Officer,Part-Time,...,Inactive,Bachelor,pratigya.sharma42@company.com,9897299598,9897403192,Rita Thapa,Power BI,Evening,Excellent,Yes
3,43,EMP0043,Amit,Sharma,Female,23,Chitwan,Finance,Accountant,Intern,...,Active,Master,amit.sharma43@company.com,9817143479,9882701840,Mina Rai,AWS,Day,Excellent,No
4,44,EMP0044,Suman,Shrestha,Male,49,Kathmandu,Marketing,Marketing Executive,Contract,...,Active,Master,suman.shrestha44@company.com,9896723552,9897990221,Prakash Karki,Python,Flexible,Average,Yes
5,49,EMP0049,Roshani,Sharma,Other,41,Chitwan,Analytics,BI Analyst,Contract,...,On Leave,Diploma,roshani.sharma49@company.com,9892734355,9883696064,Rita Thapa,AWS,Day,Excellent,No
6,60,EMP0060,Roshani,Shrestha,Other,27,Butwal,IT,Data Engineer,Part-Time,...,Active,Diploma,roshani.shrestha60@company.com,9864888845,9859795676,Prakash Karki,PMP,Evening,Excellent,No
7,63,EMP0063,Suman,Shrestha,Other,55,Kathmandu,Sales,Sales Manager,Contract,...,Inactive,Diploma,suman.shrestha63@company.com,9836063000,9884127728,Rita Thapa,AWS,Evening,Excellent,Yes
8,65,EMP0065,Roshani,Shrestha,Female,34,Biratnagar,Analytics,Data Analyst,Full-Time,...,On Leave,Diploma,roshani.shrestha65@company.com,9848110898,9850083036,Prakash Karki,NaN,Day,Excellent,Yes
9,72,EMP0072,Amit,Sharma,Other,38,Kathmandu,Finance,Finance Officer,Part-Time,...,On Leave,PhD,amit.sharma72@company.com,9871713926,9879496093,Prakash Karki,NaN,Evening,Excellent,Yes


**Q23. Find employees whose job_title contains the word Analyst.**

In [35]:
query_23 = """
SELECT *
FROM employees
WHERE job_title LIKE '%%Analyst%%';
"""
result_23 = run_query(query_23)
print(f"Rows returned: {len(result_23)}")
result_23.head(20)


Rows returned: 23


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
1,18,EMP0018,Mina,Adhikari,Male,41,Butwal,Finance,Financial Analyst,Contract,...,On Leave,Master,mina.adhikari18@company.com,9830246318,9863562849,Amit Pandey,NaN,Evening,Good,No
2,30,EMP0030,Anish,Basnet,Male,53,Pokhara,Analytics,Data Analyst,Part-Time,...,On Leave,Diploma,anish.basnet30@company.com,9852672583,9898658609,Mina Rai,AWS,Flexible,Excellent,Yes
3,32,EMP0032,Pratigya,Karki,Male,57,Pokhara,Analytics,Data Analyst,Full-Time,...,Active,Master,pratigya.karki32@company.com,9882520415,9857868867,Prakash Karki,Python,Evening,Average,No
4,37,EMP0037,Kusum,Sharma,Other,26,Kathmandu,Analytics,Data Analyst,Intern,...,Inactive,Bachelor,kusum.sharma37@company.com,9849830184,9814996960,Prakash Karki,SQL,Evening,Average,No
5,39,EMP0039,Roshani,Maharjan,Female,50,Bhaktapur,Finance,Financial Analyst,Full-Time,...,Inactive,Diploma,roshani.maharjan39@company.com,9843449924,9889951966,Mina Rai,NaN,Flexible,Good,No
6,48,EMP0048,Ramesh,Rai,Other,49,Kathmandu,Analytics,Data Analyst,Part-Time,...,Inactive,PhD,ramesh.rai48@company.com,9832568994,9823644673,Prakash Karki,PMP,Flexible,Good,Yes
7,49,EMP0049,Roshani,Sharma,Other,41,Chitwan,Analytics,BI Analyst,Contract,...,On Leave,Diploma,roshani.sharma49@company.com,9892734355,9883696064,Rita Thapa,AWS,Day,Excellent,No
8,51,EMP0051,Nabin,Poudel,Other,60,Lalitpur,Analytics,Data Analyst,Intern,...,On Leave,Bachelor,nabin.poudel51@company.com,9867614636,NaN,Amit Pandey,Power BI,Day,Excellent,Yes
9,53,EMP0053,Sneha,Khatri,Female,27,Kathmandu,Finance,Financial Analyst,Contract,...,Active,Master,sneha.khatri53@company.com,9875191287,9855752567,Amit Pandey,PMP,Evening,Good,Yes


**Q24. Find employees whose email ends with @company.com.**

In [37]:
query_24 = """
SELECT *
FROM employees
WHERE email LIKE '%%@company.com';
"""
result_24 = run_query(query_24)
print(f"Rows returned: {len(result_24)}")
result_24.head(len(result_24))


Rows returned: 143


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
3,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
4,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
138,146,EMP0146,Anita,Karki,Female,28,Chitwan,Operations,Operations Officer,Contract,...,Active,Master,anita.karki146@company.com,9893718200,9854339328,Sanjay Sharma,Power BI,Flexible,Average,No
139,147,EMP0147,Mina,Lama,Other,40,Biratnagar,HR,HR Officer,Full-Time,...,On Leave,Bachelor,mina.lama147@company.com,9883240890,9845164663,Mina Rai,PMP,Evening,Average,Yes
140,148,EMP0148,Aarav,Tamang,Other,35,Lalitpur,Sales,Sales Manager,Contract,...,Active,Bachelor,aarav.tamang148@company.com,9817882426,9815629239,Rita Thapa,Python,Day,Good,Yes
141,149,EMP0149,Sabina,KC,Male,39,Bhaktapur,Analytics,Data Analyst,Full-Time,...,Active,Bachelor,sabina.kc149@company.com,9840955018,9841011816,Prakash Karki,Power BI,Day,Excellent,No


### IN Operator

**Q25. Find employees working in Kathmandu, Pokhara, or Lalitpur.**

In [38]:
query_25 = """
SELECT *
FROM employees
WHERE city IN ('Kathmandu', 'Pokhara', 'Lalitpur');
"""
result_25 = run_query(query_25)
print(f"Rows returned: {len(result_25)}")
result_25.head(20)


Rows returned: 63


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
2,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
3,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
4,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes
5,23,EMP0023,Sanjay,Maharjan,Male,52,Lalitpur,Finance,Finance Officer,Part-Time,...,Active,PhD,sanjay.maharjan23@company.com,NaN,9821793294,Sanjay Sharma,Power BI,Evening,Average,No
6,26,EMP0026,Nabin,Karki,Other,52,Lalitpur,IT,Software Engineer,Contract,...,Inactive,Diploma,nabin.karki26@company.com,9845843496,9885017167,Sanjay Sharma,NaN,Evening,Good,No
7,28,EMP0028,Amit,Sharma,Female,31,Pokhara,HR,HR Manager,Part-Time,...,On Leave,PhD,amit.sharma28@company.com,9813798947,9846356183,Sanjay Sharma,Power BI,Evening,Good,No
8,30,EMP0030,Anish,Basnet,Male,53,Pokhara,Analytics,Data Analyst,Part-Time,...,On Leave,Diploma,anish.basnet30@company.com,9852672583,9898658609,Mina Rai,AWS,Flexible,Excellent,Yes
9,32,EMP0032,Pratigya,Karki,Male,57,Pokhara,Analytics,Data Analyst,Full-Time,...,Active,Master,pratigya.karki32@company.com,9882520415,9857868867,Prakash Karki,Python,Evening,Average,No


**Q26. Find employees in the IT, Analytics, or Finance departments.**

In [39]:
query_26 = """
SELECT *
FROM employees
WHERE department IN ('IT', 'Analytics', 'Finance');
"""
result_26 = run_query(query_26)
print(f"Rows returned: {len(result_26)}")
result_26.head(20)


Rows returned: 70


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
2,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
3,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
4,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
5,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
6,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
7,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
8,18,EMP0018,Mina,Adhikari,Male,41,Butwal,Finance,Financial Analyst,Contract,...,On Leave,Master,mina.adhikari18@company.com,9830246318,9863562849,Amit Pandey,NaN,Evening,Good,No
9,20,EMP0020,Suman,Gurung,Other,58,Bhaktapur,IT,Data Engineer,Contract,...,On Leave,Bachelor,suman.gurung20@company.com,9824082320,9821480544,Amit Pandey,Python,Flexible,Average,Yes


**Q27. Find employees with employment_type Full-Time or Contract.**

In [40]:
query_27 = """
SELECT *
FROM employees
WHERE employment_type IN ('Full-Time', 'Contract');
"""
result_27 = run_query(query_27)
print(f"Rows returned: {len(result_27)}")
result_27.head(20)


Rows returned: 77


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
3,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
4,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
5,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
6,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
7,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
8,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
9,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes


**Q28. Find employees whose education_level is Bachelor, Master, or PhD.**

In [41]:
query_28 = """
SELECT *
FROM employees
WHERE education_level IN ('Bachelor', 'Master', 'PhD');
"""
result_28 = run_query(query_28)
print(f"Rows returned: {len(result_28)}")
result_28.head(20)


Rows returned: 118


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
1,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
2,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
3,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
4,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
5,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
6,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes
7,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
8,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
9,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No


### BETWEEN Operator

**Q29. Find employees aged between 25 and 40.**

In [42]:
query_29 = """
SELECT *
FROM employees
WHERE age BETWEEN 25 AND 40;
"""
result_29 = run_query(query_29)
print(f"Rows returned: {len(result_29)}")
result_29.head(20)


Rows returned: 53


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
1,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
2,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
3,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes
4,21,EMP0021,Sneha,Rai,Male,39,Butwal,HR,HR Manager,Part-Time,...,Inactive,Bachelor,sneha.rai21@company.com,9896943854,9874426133,Amit Pandey,Power BI,Day,Good,Yes
5,22,EMP0022,Aarav,Maharjan,Female,28,Butwal,HR,HR Manager,Full-Time,...,On Leave,PhD,aarav.maharjan22@company.com,9824263193,9853794068,Rita Thapa,Power BI,Flexible,Good,Yes
6,27,EMP0027,Aarya,Karki,Other,35,Biratnagar,IT,Software Engineer,Full-Time,...,Active,Master,aarya.karki27@company.com,9894579966,9859584636,Amit Pandey,SQL,Evening,Excellent,No
7,28,EMP0028,Amit,Sharma,Female,31,Pokhara,HR,HR Manager,Part-Time,...,On Leave,PhD,amit.sharma28@company.com,9813798947,9846356183,Sanjay Sharma,Power BI,Evening,Good,No
8,31,EMP0031,Sabina,Gurung,Other,34,Bhaktapur,IT,QA Engineer,Part-Time,...,Active,Master,sabina.gurung31@company.com,9885012599,9853722847,Rita Thapa,AWS,Flexible,Excellent,Yes
9,33,EMP0033,Aarav,Rai,Female,35,Kathmandu,Sales,Sales Executive,Contract,...,Inactive,PhD,aarav.rai33@company.com,9878214548,9815405350,Rita Thapa,Power BI,Evening,Excellent,Yes


**Q30. Find employees whose monthly_salary is between 80000 and 150000.**

In [43]:
query_30 = """
SELECT *
FROM employees
WHERE monthly_salary BETWEEN 80000 AND 150000;
"""
result_30 = run_query(query_30)
print(f"Rows returned: {len(result_30)}")
result_30.head(20)


Rows returned: 45


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
3,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
4,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,NaN,Mina Rai,AWS,Flexible,Average,No
5,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes
6,21,EMP0021,Sneha,Rai,Male,39,Butwal,HR,HR Manager,Part-Time,...,Inactive,Bachelor,sneha.rai21@company.com,9896943854,9874426133,Amit Pandey,Power BI,Day,Good,Yes
7,24,EMP0024,Deepak,Maharjan,Female,52,Bhaktapur,Finance,Finance Officer,Part-Time,...,On Leave,PhD,deepak.maharjan24@company.com,9889170107,9861221192,Mina Rai,Python,Evening,Excellent,No
8,25,EMP0025,Pratigya,Poudel,Other,49,Bhaktapur,Finance,Accountant,Full-Time,...,Inactive,PhD,pratigya.poudel25@company.com,9849388796,9887093109,Prakash Karki,Power BI,Flexible,Good,No
9,29,EMP0029,Sanjay,Rai,Male,52,Chitwan,Sales,Business Development Officer,Intern,...,On Leave,Master,sanjay.rai29@company.com,9872561166,9836673639,Prakash Karki,AWS,Flexible,Excellent,No


**Q31. Find employees with performance_rating between 3.5 and 4.5.**

In [44]:
query_31 = """
SELECT *
FROM employees
WHERE performance_rating BETWEEN 3.5 AND 4.5;
"""
result_31 = run_query(query_31)
print(f"Rows returned: {len(result_31)}")
result_31.head(20)


Rows returned: 72


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
3,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
4,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
5,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
6,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
7,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
8,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
9,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No


**Q32. Find employees with years_experience between 3 and 10.**

In [45]:
query_32 = """
SELECT *
FROM employees
WHERE years_experience BETWEEN 3 AND 10;
"""
result_32 = run_query(query_32)
print(f"Rows returned: {len(result_32)}")
result_32.head(20)


Rows returned: 97


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
3,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
4,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
5,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
6,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
7,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes
8,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
9,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes


**Q33. Find employees who joined between 2020-01-01 and 2024-12-31.**

In [46]:
query_33 = """
SELECT *
FROM employees
WHERE join_date BETWEEN '2020-01-01' AND '2024-12-31';
"""
result_33 = run_query(query_33)
print(f"Rows returned: {len(result_33)}")
result_33.head(20)


Rows returned: 63


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
1,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
2,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
3,14,EMP0014,Rita,Poudel,Male,37,Chitwan,IT,QA Engineer,Full-Time,...,On Leave,PhD,rita.poudel14@company.com,9887373976,9857017385,Amit Pandey,AWS,Flexible,Excellent,No
4,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
5,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,NaN,Mina Rai,AWS,Flexible,Average,No
6,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes
7,20,EMP0020,Suman,Gurung,Other,58,Bhaktapur,IT,Data Engineer,Contract,...,On Leave,Bachelor,suman.gurung20@company.com,9824082320,9821480544,Amit Pandey,Python,Flexible,Average,Yes
8,21,EMP0021,Sneha,Rai,Male,39,Butwal,HR,HR Manager,Part-Time,...,Inactive,Bachelor,sneha.rai21@company.com,9896943854,9874426133,Amit Pandey,Power BI,Day,Good,Yes
9,24,EMP0024,Deepak,Maharjan,Female,52,Bhaktapur,Finance,Finance Officer,Part-Time,...,On Leave,PhD,deepak.maharjan24@company.com,9889170107,9861221192,Mina Rai,Python,Evening,Excellent,No


### IS NULL / IS NOT NULL

**Q34. Find employees whose email is NULL.**

In [47]:
query_34 = """
SELECT *
FROM employees
WHERE email IS NULL;
"""
result_34 = run_query(query_34)
print(f"Rows returned: {len(result_34)}")
result_34.head(20)


Rows returned: 7


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,None,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes
1,38,EMP0038,Sanjay,Adhikari,Other,22,Kathmandu,Sales,Business Development Officer,Contract,...,Inactive,Master,None,9816199453,9819546277,Sanjay Sharma,PMP,Evening,Good,Yes
2,57,EMP0057,Anita,Poudel,Other,59,Bhaktapur,Finance,Finance Officer,Intern,...,Inactive,Bachelor,None,9864493103,9896832664,Prakash Karki,Power BI,Evening,Good,No
3,76,EMP0076,Anita,Tamang,Female,43,Butwal,Marketing,Marketing Executive,Intern,...,On Leave,Bachelor,None,9815670826,9878610191,Mina Rai,NaN,Evening,Good,Yes
4,95,EMP0095,Mina,Karki,Female,59,Biratnagar,Marketing,Content Specialist,Intern,...,Inactive,Diploma,None,9839052337,9832589858,Rita Thapa,SQL,Flexible,Excellent,Yes
5,114,EMP0114,Aarya,Lama,Other,47,Kathmandu,Analytics,Data Scientist,Part-Time,...,Inactive,Bachelor,None,9838411370,9836421875,Prakash Karki,AWS,Flexible,Good,No
6,133,EMP0133,Aarya,Rai,Male,34,Pokhara,Finance,Finance Officer,Intern,...,On Leave,Master,None,9866873653,9816280106,Amit Pandey,Power BI,Evening,Excellent,Yes


**Q35. Find employees whose phone is NULL.**

In [48]:
query_35 = """
SELECT *
FROM employees
WHERE phone IS NULL;
"""
result_35 = run_query(query_35)
print(f"Rows returned: {len(result_35)}")
result_35.head(20)


Rows returned: 6


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,23,EMP0023,Sanjay,Maharjan,Male,52,Lalitpur,Finance,Finance Officer,Part-Time,...,Active,PhD,sanjay.maharjan23@company.com,None,9821793294,Sanjay Sharma,Power BI,Evening,Average,No
1,46,EMP0046,Ramesh,Maharjan,Male,36,Chitwan,IT,Software Engineer,Part-Time,...,Inactive,Diploma,ramesh.maharjan46@company.com,None,9833557996,Sanjay Sharma,NaN,Day,Average,No
2,69,EMP0069,Binita,Basnet,Male,59,Butwal,Analytics,Data Analyst,Intern,...,On Leave,PhD,binita.basnet69@company.com,None,9863222909,Rita Thapa,Power BI,Flexible,Excellent,No
3,92,EMP0092,Rita,Thapa,Female,32,Kathmandu,Sales,Business Development Officer,Intern,...,On Leave,Bachelor,rita.thapa92@company.com,None,9821869890,Amit Pandey,SQL,Day,Excellent,Yes
4,115,EMP0115,Sanjay,Rai,Other,39,Biratnagar,Analytics,BI Analyst,Part-Time,...,On Leave,PhD,sanjay.rai115@company.com,None,9830089243,Rita Thapa,SQL,Day,Excellent,No
5,138,EMP0138,Pooja,Khatri,Female,32,Kathmandu,IT,QA Engineer,Full-Time,...,Active,Master,pooja.khatri138@company.com,None,9814856855,Amit Pandey,NaN,Day,Excellent,Yes


**Q36. Find employees whose emergency_contact is NULL.**

In [49]:
query_36 = """
SELECT *
FROM employees
WHERE emergency_contact IS NULL;
"""
result_36 = run_query(query_36)
print(f"Rows returned: {len(result_36)}")
result_36.head(20)


Rows returned: 8


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,None,Mina Rai,AWS,Flexible,Average,No
1,34,EMP0034,Binita,Poudel,Female,26,Chitwan,Sales,Sales Manager,Intern,...,On Leave,PhD,binita.poudel34@company.com,9838549945,None,Mina Rai,AWS,Flexible,Excellent,No
2,51,EMP0051,Nabin,Poudel,Other,60,Lalitpur,Analytics,Data Analyst,Intern,...,On Leave,Bachelor,nabin.poudel51@company.com,9867614636,None,Amit Pandey,Power BI,Day,Excellent,Yes
3,68,EMP0068,Roshani,Maharjan,Female,43,Kathmandu,IT,Software Engineer,Intern,...,On Leave,Master,roshani.maharjan68@company.com,9871761686,None,Rita Thapa,NaN,Evening,Average,Yes
4,85,EMP0085,Pooja,Khatri,Female,59,Biratnagar,Sales,Sales Executive,Part-Time,...,Active,PhD,pooja.khatri85@company.com,9820233067,None,Mina Rai,SQL,Day,Excellent,Yes
5,102,EMP0102,Ramesh,Sharma,Male,57,Lalitpur,Operations,Project Coordinator,Full-Time,...,On Leave,PhD,ramesh.sharma102@company.com,9828945559,None,Prakash Karki,AWS,Flexible,Average,Yes
6,119,EMP0119,Runa,Pandey,Other,44,Lalitpur,HR,HR Officer,Contract,...,Inactive,PhD,runa.pandey119@company.com,9846081313,None,Mina Rai,AWS,Flexible,Average,Yes
7,136,EMP0136,Prakash,Maharjan,Male,22,Pokhara,Operations,Operations Manager,Full-Time,...,On Leave,Master,prakash.maharjan136@company.com,9846381299,None,Sanjay Sharma,Python,Day,Good,Yes


**Q37. Find employees whose certification is NULL.**

In [50]:
query_37 = """
SELECT *
FROM employees
WHERE certification IS NULL;
"""
result_37 = run_query(query_37)
print(f"Rows returned: {len(result_37)}")
result_37.head(20)


Rows returned: 35


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,None,Evening,Good,Yes
1,18,EMP0018,Mina,Adhikari,Male,41,Butwal,Finance,Financial Analyst,Contract,...,On Leave,Master,mina.adhikari18@company.com,9830246318,9863562849,Amit Pandey,None,Evening,Good,No
2,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,None,Day,Average,Yes
3,26,EMP0026,Nabin,Karki,Other,52,Lalitpur,IT,Software Engineer,Contract,...,Inactive,Diploma,nabin.karki26@company.com,9845843496,9885017167,Sanjay Sharma,None,Evening,Good,No
4,39,EMP0039,Roshani,Maharjan,Female,50,Bhaktapur,Finance,Financial Analyst,Full-Time,...,Inactive,Diploma,roshani.maharjan39@company.com,9843449924,9889951966,Mina Rai,None,Flexible,Good,No
5,45,EMP0045,Pratigya,Pandey,Other,30,Lalitpur,Analytics,Data Scientist,Contract,...,Active,Bachelor,pratigya.pandey45@company.com,9894458990,9866187470,Mina Rai,None,Flexible,Good,Yes
6,46,EMP0046,Ramesh,Maharjan,Male,36,Chitwan,IT,Software Engineer,Part-Time,...,Inactive,Diploma,ramesh.maharjan46@company.com,NaN,9833557996,Sanjay Sharma,None,Day,Average,No
7,47,EMP0047,Anish,Thapa,Female,21,Bhaktapur,IT,QA Engineer,Full-Time,...,Active,Master,anish.thapa47@company.com,9861155566,9895030117,Rita Thapa,None,Evening,Good,Yes
8,50,EMP0050,Sujan,Lama,Female,33,Butwal,IT,QA Engineer,Contract,...,Inactive,PhD,sujan.lama50@company.com,9892119212,9851221953,Sanjay Sharma,None,Evening,Excellent,No
9,52,EMP0052,Sanjay,Lama,Male,46,Kathmandu,HR,Recruiter,Contract,...,Inactive,Bachelor,sanjay.lama52@company.com,9825693263,9839171865,Sanjay Sharma,None,Evening,Excellent,No


**Q38. Find employees whose email is NOT NULL AND phone is NOT NULL.**

In [51]:
query_38 = """
SELECT *
FROM employees
WHERE email IS NOT NULL AND phone IS NOT NULL;
"""
result_38 = run_query(query_38)
print(f"Rows returned: {len(result_38)}")
result_38.head(20)


Rows returned: 137


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
3,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
4,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
5,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
6,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
7,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
8,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
9,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes


### Mixed Challenge

**Q39. Find Active employees from Kathmandu or Lalitpur whose salary is between 90000 and 180000.**

In [52]:
query_39 = """
SELECT *
FROM employees
WHERE employment_status = 'Active'
  AND (city = 'Kathmandu' OR city = 'Lalitpur')
  AND monthly_salary BETWEEN 90000 AND 180000;
"""
result_39 = run_query(query_39)
print(f"Rows returned: {len(result_39)}")
result_39.head(20)


Rows returned: 6


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,45,EMP0045,Pratigya,Pandey,Other,30,Lalitpur,Analytics,Data Scientist,Contract,...,Active,Bachelor,pratigya.pandey45@company.com,9894458990,9866187470,Mina Rai,NaN,Flexible,Good,Yes
2,53,EMP0053,Sneha,Khatri,Female,27,Kathmandu,Finance,Financial Analyst,Contract,...,Active,Master,sneha.khatri53@company.com,9875191287,9855752567,Amit Pandey,PMP,Evening,Good,Yes
3,78,EMP0078,Sunita,Poudel,Male,59,Kathmandu,Analytics,BI Analyst,Intern,...,Active,PhD,sunita.poudel78@company.com,9844867731,9883134170,Prakash Karki,NaN,Day,Good,No
4,105,EMP0105,Sabina,Pandey,Other,49,Lalitpur,Finance,Financial Analyst,Intern,...,Active,Master,sabina.pandey105@company.com,9811487240,9851381739,Amit Pandey,NaN,Flexible,Average,No
5,110,EMP0110,Sabina,Shrestha,Other,52,Kathmandu,Sales,Sales Manager,Part-Time,...,Active,Bachelor,sabina.shrestha110@company.com,9828642937,9811754847,Rita Thapa,SQL,Day,Good,No


**Q40. Find employees in IT or Analytics whose first_name starts with A and performance_rating is at least 4.**

In [54]:
query_40 = """
SELECT *
FROM employees
WHERE (department = 'IT' OR department = 'Analytics')
  AND first_name LIKE 'A%%'
  AND performance_rating >= 4;
"""
result_40 = run_query(query_40)
print(f"Rows returned: {len(result_40)}")
result_40.head(20)


Rows returned: 3


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,47,EMP0047,Anish,Thapa,Female,21,Bhaktapur,IT,QA Engineer,Full-Time,...,Active,Master,anish.thapa47@company.com,9861155566,9895030117,Rita Thapa,NaN,Evening,Good,Yes
1,74,EMP0074,Aarya,Gurung,Other,59,Bhaktapur,Analytics,BI Analyst,Full-Time,...,On Leave,Diploma,aarya.gurung74@company.com,9814857511,9851846762,Rita Thapa,PMP,Day,Excellent,No
2,80,EMP0080,Arjun,Adhikari,Female,31,Biratnagar,Analytics,BI Analyst,Part-Time,...,On Leave,Master,arjun.adhikari80@company.com,9830715403,9894736966,Prakash Karki,Python,Flexible,Excellent,No


**Q41. Find employees who are not Interns and have completed more than 5 projects.**

In [55]:
query_41 = """
SELECT *
FROM employees
WHERE employment_type <> 'Intern'
  AND projects_completed > 5;
"""
result_41 = run_query(query_41)
print(f"Rows returned: {len(result_41)}")
result_41.head(20)


Rows returned: 70


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
3,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
4,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
5,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
6,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
7,12,EMP0012,Mina,Tamang,Other,47,Biratnagar,Marketing,Marketing Executive,Contract,...,Inactive,PhD,mina.tamang12@company.com,9893320274,9847644351,Sanjay Sharma,PMP,Flexible,Average,Yes
8,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
9,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No


**Q42. Find employees with NULL certification OR NULL emergency_contact.**

In [56]:
query_42 = """
SELECT *
FROM employees
WHERE certification IS NULL OR emergency_contact IS NULL;
"""
result_42 = run_query(query_42)
print(f"Rows returned: {len(result_42)}")
result_42.head(20)


Rows returned: 42


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
1,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,NaN,Mina Rai,AWS,Flexible,Average,No
2,18,EMP0018,Mina,Adhikari,Male,41,Butwal,Finance,Financial Analyst,Contract,...,On Leave,Master,mina.adhikari18@company.com,9830246318,9863562849,Amit Pandey,NaN,Evening,Good,No
3,19,EMP0019,Sabina,Adhikari,Female,51,Kathmandu,Operations,Project Coordinator,Contract,...,Inactive,PhD,NaN,9864804706,9815034008,Amit Pandey,NaN,Day,Average,Yes
4,26,EMP0026,Nabin,Karki,Other,52,Lalitpur,IT,Software Engineer,Contract,...,Inactive,Diploma,nabin.karki26@company.com,9845843496,9885017167,Sanjay Sharma,NaN,Evening,Good,No
5,34,EMP0034,Binita,Poudel,Female,26,Chitwan,Sales,Sales Manager,Intern,...,On Leave,PhD,binita.poudel34@company.com,9838549945,NaN,Mina Rai,AWS,Flexible,Excellent,No
6,39,EMP0039,Roshani,Maharjan,Female,50,Bhaktapur,Finance,Financial Analyst,Full-Time,...,Inactive,Diploma,roshani.maharjan39@company.com,9843449924,9889951966,Mina Rai,NaN,Flexible,Good,No
7,45,EMP0045,Pratigya,Pandey,Other,30,Lalitpur,Analytics,Data Scientist,Contract,...,Active,Bachelor,pratigya.pandey45@company.com,9894458990,9866187470,Mina Rai,NaN,Flexible,Good,Yes
8,46,EMP0046,Ramesh,Maharjan,Male,36,Chitwan,IT,Software Engineer,Part-Time,...,Inactive,Diploma,ramesh.maharjan46@company.com,NaN,9833557996,Sanjay Sharma,NaN,Day,Average,No
9,47,EMP0047,Anish,Thapa,Female,21,Bhaktapur,IT,QA Engineer,Full-Time,...,Active,Master,anish.thapa47@company.com,9861155566,9895030117,Rita Thapa,NaN,Evening,Good,Yes


**Q43. Find employees whose job_title contains Manager and whose employment_status is Active.**

In [57]:
query_43 = """
SELECT *
FROM employees
WHERE job_title LIKE '%%Manager%%'
  AND employment_status = 'Active';
"""
result_43 = run_query(query_43)
print(f"Rows returned: {len(result_43)}")
result_43.head(20)


Rows returned: 11


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,NaN,Mina Rai,AWS,Flexible,Average,No
1,41,EMP0041,Anish,Adhikari,Female,31,Chitwan,Sales,Sales Manager,Contract,...,Active,PhD,anish.adhikari41@company.com,9836393021,9816860253,Rita Thapa,AWS,Day,Excellent,Yes
2,56,EMP0056,Sujan,Poudel,Other,59,Butwal,HR,HR Manager,Intern,...,Active,PhD,sujan.poudel56@company.com,9865486797,9844977265,Prakash Karki,Python,Flexible,Good,No
3,62,EMP0062,Prakash,Thapa,Female,39,Lalitpur,HR,HR Manager,Part-Time,...,Active,PhD,prakash.thapa62@company.com,9859264105,9887444334,Sanjay Sharma,NaN,Evening,Excellent,No
4,81,EMP0081,Aarav,Karki,Female,59,Bhaktapur,Operations,Operations Manager,Full-Time,...,Active,Master,aarav.karki81@company.com,9850081864,9826393507,Sanjay Sharma,AWS,Day,Average,Yes
5,96,EMP0096,Pratigya,Shrestha,Female,35,Biratnagar,Marketing,Brand Manager,Contract,...,Active,Master,pratigya.shrestha96@company.com,9899228205,9813690274,Prakash Karki,NaN,Day,Good,No
6,110,EMP0110,Sabina,Shrestha,Other,52,Kathmandu,Sales,Sales Manager,Part-Time,...,Active,Bachelor,sabina.shrestha110@company.com,9828642937,9811754847,Rita Thapa,SQL,Day,Good,No
7,112,EMP0112,Sabina,Shrestha,Male,51,Biratnagar,HR,HR Manager,Intern,...,Active,Diploma,sabina.shrestha112@company.com,9853529432,9887182438,Sanjay Sharma,Python,Evening,Good,No
8,126,EMP0126,Sita,Lama,Other,28,Bhaktapur,Marketing,Brand Manager,Intern,...,Active,Master,sita.lama126@company.com,9862211642,9852134402,Rita Thapa,NaN,Evening,Average,Yes
9,141,EMP0141,Sneha,KC,Male,28,Biratnagar,Marketing,Brand Manager,Full-Time,...,Active,PhD,sneha.kc141@company.com,9827691817,9854391749,Amit Pandey,SQL,Evening,Good,No


**Q44. Display employees aged between 30 and 50 who work remotely and have salary above 120000.**

In [58]:
query_44 = """
SELECT *
FROM employees
WHERE age BETWEEN 30 AND 50
  AND remote_worker = 'Yes'
  AND monthly_salary > 120000;
"""
result_44 = run_query(query_44)
print(f"Rows returned: {len(result_44)}")
result_44.head(20)


Rows returned: 25


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
1,18,EMP0018,Mina,Adhikari,Male,41,Butwal,Finance,Financial Analyst,Contract,...,On Leave,Master,mina.adhikari18@company.com,9830246318,9863562849,Amit Pandey,NaN,Evening,Good,No
2,21,EMP0021,Sneha,Rai,Male,39,Butwal,HR,HR Manager,Part-Time,...,Inactive,Bachelor,sneha.rai21@company.com,9896943854,9874426133,Amit Pandey,Power BI,Day,Good,Yes
3,39,EMP0039,Roshani,Maharjan,Female,50,Bhaktapur,Finance,Financial Analyst,Full-Time,...,Inactive,Diploma,roshani.maharjan39@company.com,9843449924,9889951966,Mina Rai,NaN,Flexible,Good,No
4,41,EMP0041,Anish,Adhikari,Female,31,Chitwan,Sales,Sales Manager,Contract,...,Active,PhD,anish.adhikari41@company.com,9836393021,9816860253,Rita Thapa,AWS,Day,Excellent,Yes
5,46,EMP0046,Ramesh,Maharjan,Male,36,Chitwan,IT,Software Engineer,Part-Time,...,Inactive,Diploma,ramesh.maharjan46@company.com,NaN,9833557996,Sanjay Sharma,NaN,Day,Average,No
6,67,EMP0067,Mina,Tamang,Female,44,Biratnagar,Operations,Operations Manager,Full-Time,...,On Leave,Master,mina.tamang67@company.com,9868167933,9872507834,Sanjay Sharma,NaN,Flexible,Good,No
7,73,EMP0073,Anish,Lama,Female,44,Kathmandu,Marketing,Marketing Executive,Intern,...,Active,Bachelor,anish.lama73@company.com,9894059886,9852075442,Sanjay Sharma,NaN,Flexible,Excellent,No
8,77,EMP0077,Roshani,Lama,Other,31,Butwal,Sales,Sales Executive,Intern,...,Active,PhD,roshani.lama77@company.com,9818242695,9827301329,Mina Rai,Power BI,Flexible,Excellent,Yes
9,80,EMP0080,Arjun,Adhikari,Female,31,Biratnagar,Analytics,BI Analyst,Part-Time,...,On Leave,Master,arjun.adhikari80@company.com,9830715403,9894736966,Prakash Karki,Python,Flexible,Excellent,No


**Q45. Calculate annual_salary and total_compensation for employees in Finance, IT, and Analytics.**

In [59]:
query_45 = """
SELECT employee_id, department, monthly_salary, annual_bonus,
       monthly_salary * 12 AS annual_salary,
       (monthly_salary * 12) + annual_bonus AS total_compensation
FROM employees
WHERE department IN ('Finance', 'IT', 'Analytics');
"""
result_45 = run_query(query_45)
print(f"Rows returned: {len(result_45)}")
result_45.head(20)


Rows returned: 70


,employee_id,department,monthly_salary,annual_bonus,annual_salary,total_compensation
0,1,IT,105000.0,15750.0,1260000.0,1275750.0
1,3,IT,70000.0,8400.0,840000.0,848400.0
2,5,IT,200000.0,24000.0,2400000.0,2424000.0
3,7,Analytics,35000.0,3500.0,420000.0,423500.0
4,8,Analytics,195000.0,23400.0,2340000.0,2363400.0
5,10,IT,240000.0,19200.0,2880000.0,2899200.0
6,13,IT,87500.0,4375.0,1050000.0,1054375.0
7,14,IT,37500.0,1875.0,450000.0,451875.0
8,18,Finance,245000.0,12250.0,2940000.0,2952250.0
9,20,IT,217500.0,10875.0,2610000.0,2620875.0


**Q46. Find employees whose promotion_eligible = Yes AND performance_category = Excellent.**

In [60]:
query_46 = """
SELECT *
FROM employees
WHERE promotion_eligible = 'Yes'
  AND performance_category = 'Excellent';
"""
result_46 = run_query(query_46)
print(f"Rows returned: {len(result_46)}")
result_46.head(20)


Rows returned: 32


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,3,EMP0003,Ramesh,Maharjan,Other,58,Bhaktapur,IT,QA Engineer,Full-Time,...,Inactive,Master,ramesh.maharjan3@company.com,9880417695,9844714533,Mina Rai,Power BI,Day,Excellent,Yes
1,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
2,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
3,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
4,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
5,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes
6,30,EMP0030,Anish,Basnet,Male,53,Pokhara,Analytics,Data Analyst,Part-Time,...,On Leave,Diploma,anish.basnet30@company.com,9852672583,9898658609,Mina Rai,AWS,Flexible,Excellent,Yes
7,31,EMP0031,Sabina,Gurung,Other,34,Bhaktapur,IT,QA Engineer,Part-Time,...,Active,Master,sabina.gurung31@company.com,9885012599,9853722847,Rita Thapa,AWS,Flexible,Excellent,Yes
8,33,EMP0033,Aarav,Rai,Female,35,Kathmandu,Sales,Sales Executive,Contract,...,Inactive,PhD,aarav.rai33@company.com,9878214548,9815405350,Rita Thapa,Power BI,Evening,Excellent,Yes
9,41,EMP0041,Anish,Adhikari,Female,31,Chitwan,Sales,Sales Manager,Contract,...,Active,PhD,anish.adhikari41@company.com,9836393021,9816860253,Rita Thapa,AWS,Day,Excellent,Yes


**Q47. Find employees with overtime_hours between 20 and 60 and leave_days_taken less than 15.**

In [61]:
query_47 = """
SELECT *
FROM employees
WHERE overtime_hours BETWEEN 20 AND 60
  AND leave_days_taken < 15;
"""
result_47 = run_query(query_47)
print(f"Rows returned: {len(result_47)}")
result_47.head(20)


Rows returned: 36


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,6,EMP0006,Deepak,Thapa,Female,48,Bhaktapur,Marketing,Brand Manager,Full-Time,...,Inactive,Master,deepak.thapa6@company.com,9870316750,9811883416,Rita Thapa,SQL,Day,Excellent,Yes
1,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
2,24,EMP0024,Deepak,Maharjan,Female,52,Bhaktapur,Finance,Finance Officer,Part-Time,...,On Leave,PhD,deepak.maharjan24@company.com,9889170107,9861221192,Mina Rai,Python,Evening,Excellent,No
3,27,EMP0027,Aarya,Karki,Other,35,Biratnagar,IT,Software Engineer,Full-Time,...,Active,Master,aarya.karki27@company.com,9894579966,9859584636,Amit Pandey,SQL,Evening,Excellent,No
4,34,EMP0034,Binita,Poudel,Female,26,Chitwan,Sales,Sales Manager,Intern,...,On Leave,PhD,binita.poudel34@company.com,9838549945,NaN,Mina Rai,AWS,Flexible,Excellent,No
5,38,EMP0038,Sanjay,Adhikari,Other,22,Kathmandu,Sales,Business Development Officer,Contract,...,Inactive,Master,NaN,9816199453,9819546277,Sanjay Sharma,PMP,Evening,Good,Yes
6,39,EMP0039,Roshani,Maharjan,Female,50,Bhaktapur,Finance,Financial Analyst,Full-Time,...,Inactive,Diploma,roshani.maharjan39@company.com,9843449924,9889951966,Mina Rai,NaN,Flexible,Good,No
7,40,EMP0040,Aarav,Pandey,Male,34,Chitwan,Sales,Business Development Officer,Part-Time,...,Inactive,Bachelor,aarav.pandey40@company.com,9867943072,9887892505,Prakash Karki,SQL,Flexible,Good,Yes
8,41,EMP0041,Anish,Adhikari,Female,31,Chitwan,Sales,Sales Manager,Contract,...,Active,PhD,anish.adhikari41@company.com,9836393021,9816860253,Rita Thapa,AWS,Day,Excellent,Yes
9,45,EMP0045,Pratigya,Pandey,Other,30,Lalitpur,Analytics,Data Scientist,Contract,...,Active,Bachelor,pratigya.pandey45@company.com,9894458990,9866187470,Mina Rai,NaN,Flexible,Good,Yes


**Q48. Find employees from cities other than Kathmandu whose first_name contains the letter u.**

In [62]:
query_48 = """
SELECT *
FROM employees
WHERE city <> 'Kathmandu'
  AND first_name LIKE '%%u%%';
"""
result_48 = run_query(query_48)
print(f"Rows returned: {len(result_48)}")
result_48.head(20)


Rows returned: 18


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,7,EMP0007,Suman,Rai,Female,43,Chitwan,Analytics,Data Scientist,Contract,...,Active,Diploma,suman.rai7@company.com,9839920845,9873482774,Amit Pandey,PMP,Flexible,Excellent,Yes
1,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
2,13,EMP0013,Runa,Rai,Other,27,Butwal,IT,Software Engineer,Contract,...,On Leave,Master,runa.rai13@company.com,9852084090,9847149597,Amit Pandey,NaN,Evening,Good,Yes
3,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes
4,17,EMP0017,Suman,Khatri,Female,55,Biratnagar,HR,HR Manager,Full-Time,...,Active,Bachelor,suman.khatri17@company.com,9845408799,NaN,Mina Rai,AWS,Flexible,Average,No
5,20,EMP0020,Suman,Gurung,Other,58,Bhaktapur,IT,Data Engineer,Contract,...,On Leave,Bachelor,suman.gurung20@company.com,9824082320,9821480544,Amit Pandey,Python,Flexible,Average,Yes
6,35,EMP0035,Sujan,Pandey,Female,56,Biratnagar,Operations,Project Coordinator,Part-Time,...,On Leave,Bachelor,sujan.pandey35@company.com,9831215328,9825454771,Amit Pandey,Python,Flexible,Excellent,No
7,36,EMP0036,Sujan,Tamang,Female,46,Lalitpur,Finance,Accountant,Intern,...,Inactive,Bachelor,sujan.tamang36@company.com,9824423415,9829326305,Rita Thapa,PMP,Flexible,Average,No
8,50,EMP0050,Sujan,Lama,Female,33,Butwal,IT,QA Engineer,Contract,...,Inactive,PhD,sujan.lama50@company.com,9892119212,9851221953,Sanjay Sharma,NaN,Evening,Excellent,No
9,56,EMP0056,Sujan,Poudel,Other,59,Butwal,HR,HR Manager,Intern,...,Active,PhD,sujan.poudel56@company.com,9865486797,9844977265,Prakash Karki,Python,Flexible,Good,No


**Q49. Find employees with monthly_salary > 100000 OR annual_bonus > 150000.**

In [63]:
query_49 = """
SELECT *
FROM employees
WHERE monthly_salary > 100000 OR annual_bonus > 150000;
"""
result_49 = run_query(query_49)
print(f"Rows returned: {len(result_49)}")
result_49.head(20)


Rows returned: 97


,employee_id,employee_code,first_name,last_name,gender,age,city,department,job_title,employment_type,...,employment_status,education_level,email,phone,emergency_contact,manager_name,certification,work_shift,performance_category,promotion_eligible
0,1,EMP0001,Prakash,Adhikari,Other,53,Kathmandu,IT,Software Engineer,Contract,...,Active,Diploma,prakash.adhikari1@company.com,9869202768,9842203407,Amit Pandey,SQL,Day,Average,No
1,2,EMP0002,Prakash,Gurung,Male,46,Bhaktapur,Operations,Operations Officer,Contract,...,Active,PhD,prakash.gurung2@company.com,9884819744,9882217824,Sanjay Sharma,AWS,Evening,Good,No
2,4,EMP0004,Amit,KC,Other,51,Lalitpur,Operations,Operations Manager,Intern,...,On Leave,Diploma,amit.kc4@company.com,9843689519,9885134554,Amit Pandey,Power BI,Day,Good,No
3,5,EMP0005,Aarya,Thapa,Other,59,Kathmandu,IT,Data Engineer,Contract,...,On Leave,Bachelor,aarya.thapa5@company.com,9876727300,9868281295,Amit Pandey,PMP,Evening,Average,No
4,8,EMP0008,Runa,Maharjan,Female,58,Butwal,Analytics,BI Analyst,Contract,...,Active,Bachelor,runa.maharjan8@company.com,9824245056,9862257400,Sanjay Sharma,SQL,Flexible,Good,No
5,9,EMP0009,Nisha,Pandey,Other,41,Pokhara,Operations,Operations Manager,Full-Time,...,Inactive,Diploma,nisha.pandey9@company.com,9893663689,9855153375,Mina Rai,Power BI,Day,Excellent,Yes
6,10,EMP0010,Mina,Adhikari,Female,24,Biratnagar,IT,Software Engineer,Intern,...,On Leave,Diploma,mina.adhikari10@company.com,9839356575,9879370789,Amit Pandey,Python,Flexible,Good,Yes
7,11,EMP0011,Nisha,Thapa,Male,43,Bhaktapur,Marketing,Marketing Executive,Contract,...,Active,Master,nisha.thapa11@company.com,9833751475,9853726695,Mina Rai,PMP,Day,Excellent,Yes
8,15,EMP0015,Nabin,Maharjan,Other,28,Butwal,Marketing,Marketing Executive,Contract,...,On Leave,PhD,nabin.maharjan15@company.com,9847773020,9854365408,Sanjay Sharma,AWS,Flexible,Excellent,No
9,16,EMP0016,Suman,Khatri,Male,28,Bhaktapur,HR,Recruiter,Intern,...,Active,Diploma,suman.khatri16@company.com,9867011675,9887594913,Sanjay Sharma,Python,Flexible,Excellent,Yes


**Q50. Display employee_id, employee_code, first_name, department, monthly_salary, and monthly_salary * 12 as annual_salary for Active employees.**

In [64]:
query_50 = """
SELECT employee_id, employee_code, first_name, department, monthly_salary,
       monthly_salary * 12 AS annual_salary
FROM employees
WHERE employment_status = 'Active';
"""
result_50 = run_query(query_50)
print(f"Rows returned: {len(result_50)}")
result_50.head(20)


Rows returned: 53


,employee_id,employee_code,first_name,department,monthly_salary,annual_salary
0,1,EMP0001,Prakash,IT,105000.0,1260000.0
1,2,EMP0002,Prakash,Operations,145000.0,1740000.0
2,7,EMP0007,Suman,Analytics,35000.0,420000.0
3,8,EMP0008,Runa,Analytics,195000.0,2340000.0
4,11,EMP0011,Nisha,Marketing,117500.0,1410000.0
5,16,EMP0016,Suman,HR,232500.0,2790000.0
6,17,EMP0017,Suman,HR,115000.0,1380000.0
7,23,EMP0023,Sanjay,Finance,212500.0,2550000.0
8,27,EMP0027,Aarya,IT,245000.0,2940000.0
9,31,EMP0031,Sabina,IT,197500.0,2370000.0


## 7. Close the connection

In [65]:
cursor.close()
conn.close()
engine.dispose()
print("Connections closed.")


Connections closed.
